In [22]:
import os
import warnings
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import lightgbm as lgb
import optuna
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import CountVectorizer

In [23]:
def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    
    warnings.filterwarnings("ignore")
    random.seed(seed)
    np.random.seed(seed)
    
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(False)
    
    print(f"Random seed set to {seed}")

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2 ** 32
    random.seed(worker_seed)
    np.random.seed(worker_seed)

In [24]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [25]:
INPUT_ROOT = "/kaggle/input/datasets/jun1801/df-2026-final"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = "/kaggle/input/models/jun1801/tacnet/pytorch/default/1/best_model.pth"

In [26]:
SEQ_LENGTH = 66
FEATURE_COLS = [f"feature_{i}" for i in range(1, SEQ_LENGTH + 1)]

ATTRIBUTE_LIST = [1, 2, 3, 4, 5, 6]
ATTRIBUTE_COLS = ["start_year", "attr_1", "attr_2", "attr_3",
                  "end_year", "attr_4", "attr_5", "attr_6"]

EMBEDDING_DIM = 256
DILATIONS = [1, 2, 4, 8]
DROPOUT_RATE = 0.3
HEADS = 4
WINDOW_SIZE = 3

BATCH_SIZE = 1024

M_LIST_METRIC = [12.0, 31.0, 99.0, 12.0, 31.0, 99.0]
W_LIST_VALS = [1.0, 1.0, 100.0, 1.0, 1.0, 100.0]

DAYS_IN_MONTH = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
DAYS_MAP = {
    1: 31, 2: 28, 3: 31, 4: 30, 5: 31, 6: 30,
    7: 31, 8: 31, 9: 30, 10: 31, 11: 30, 12: 31
}

In [27]:
def drop_duplicates_custom(x_df, y_df, feature_cols=FEATURE_COLS, name="Train"):
    len_before = len(x_df)
    keep_idx = x_df.drop_duplicates(subset=feature_cols, keep="first").index
    
    x_df = x_df.loc[keep_idx].reset_index(drop=True)
    y_df = y_df.loc[keep_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_df)} hàng trùng lặp trong {name}")
    return x_df, y_df

In [28]:
def drop_overlap_train_val(x_train, y_train, x_val, feature_cols=FEATURE_COLS):
    len_before = len(x_train)
    
    val_features_unique = x_val[feature_cols].drop_duplicates().copy()
    val_features_unique["is_in_val"] = True
    
    x_train["old_idx"] = x_train.index
    train_merged = x_train.merge(val_features_unique, on=feature_cols, how="left")
    
    clean_train_idx = train_merged[train_merged["is_in_val"].isna()]["old_idx"]
    
    x_train = x_train.loc[clean_train_idx].drop(columns=["old_idx"]).reset_index(drop=True)
    y_train = y_train.loc[clean_train_idx].reset_index(drop=True)
    
    print(f" - Loại bỏ {len_before - len(x_train)} chuỗi Train bị trùng với Val")
    return x_train, y_train

In [29]:
def validate_and_clean_dates(x_df, y_df, days_map=DAYS_MAP, name="Train"):
    max_days_start = y_df["attr_1"].map(days_map)
    invalid_start_day_mask = y_df["attr_2"] > max_days_start

    max_days_end = y_df["attr_4"].map(days_map)
    invalid_end_day_mask = y_df["attr_5"] > max_days_end

    all_invalid_mask = invalid_start_day_mask | invalid_end_day_mask
    invalid_count = all_invalid_mask.sum()

    if invalid_count > 0:
        print(f" - Phát hiện {invalid_count} hàng có lỗi ngày trong {name}")
        
        y_df.loc[invalid_start_day_mask, "attr_2"] = max_days_start[invalid_start_day_mask]
        y_df.loc[invalid_end_day_mask, "attr_5"] = max_days_end[invalid_end_day_mask]
        
    else:
        print(f" - Không có lỗi ngày tháng trong {name}")

    return x_df, y_df

In [30]:
def get_vocab_size(x_train, x_val, x_test, feature_cols=FEATURE_COLS):
    all_x_data = pd.concat([x_train[feature_cols], x_val[feature_cols]], axis=0)
    unique_ids = pd.unique(all_x_data.values.ravel())
    unique_ids = [uid for uid in unique_ids if not pd.isna(uid) and uid != 0]
    
    id_to_index = {0: 0, "UNK": 1}
    for idx, raw_id in enumerate(sorted(unique_ids), start=2):
        id_to_index[raw_id] = idx
        
    vocab_size = max(id_to_index.values()) + 1
    print(f" - Kích thước từ điển {vocab_size} (PAD=0, UNK=1)")

    map_func = np.vectorize(lambda x: id_to_index.get(x, 1))
    
    for df in [x_train, x_val, x_test]:
        df.loc[:, feature_cols] = map_func(df[feature_cols].values)

    return x_train, x_val, x_test, vocab_size

In [31]:
def process_data(x_train, y_train, x_val, y_val, x_test,
                 feature_cols=FEATURE_COLS,
                 attr_cols=ATTRIBUTE_COLS,
                 days_map=DAYS_MAP):
    
    for df in [x_train, x_val, x_test]:
        df[feature_cols] = df[feature_cols].fillna(0).astype(np.int64)
    for df in [y_train, y_val]:
        df["start_year"] = 0
        df["end_year"] = (df["attr_4"] < df["attr_1"]).astype(np.int64)
        df[attr_cols] = df[attr_cols].fillna(0).astype(np.float32)

    print("Xử lý trùng lặp nội bộ...")
    x_train, y_train = drop_duplicates_custom(x_train, y_train, feature_cols, "Train")
    x_val, y_val = drop_duplicates_custom(x_val, y_val, feature_cols, "Val")
    
    print("Xử lý trùng lặp giữa Train và Val...")
    x_train, y_train = drop_overlap_train_val(x_train, y_train, x_val, feature_cols)

    print("Xử lý lỗi ngày tháng...")
    x_train, y_train = validate_and_clean_dates(x_train, y_train, days_map, "Train")
    x_val, y_val = validate_and_clean_dates(x_val, y_val, days_map, "Val")

    print("Xây dựng từ điển và map index...")
    x_train, x_val, x_test, vocab_size = get_vocab_size(x_train, x_val, x_test, feature_cols)
    
    return x_train, y_train, x_val, y_val, x_test, vocab_size

In [32]:
class UserBehaviorDataset(Dataset):
    def __init__(self, x_df, y_df=None, 
                 feature_cols=FEATURE_COLS, attr_cols=ATTRIBUTE_COLS, augment=False):
        self.x_data = x_df[feature_cols].values
        self.augment = augment 
        
        self.has_labels = y_df is not None
        if self.has_labels:
            self.y_data = y_df[attr_cols].values

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):
        x_tensor = torch.tensor(self.x_data[idx], dtype=torch.long)
        
        if self.augment:
            actions = x_tensor[x_tensor != 0]
            n_actions = len(actions)
            total_slots = len(x_tensor) 
            
            if torch.rand(1).item() < 0.5:
                n_mask = (n_actions // 10) + 1
                possible_indices = torch.arange(n_actions - 1)
                
                if len(possible_indices) >= n_mask:
                    mask_indices = possible_indices[torch.randperm(len(possible_indices))[:n_mask]]
                    actions[mask_indices] = 0
            
            if torch.rand(1).item() < 0.5:
                new_indices = torch.randperm(total_slots)[:n_actions].sort()[0]
                dilated_x = torch.zeros_like(x_tensor)
                dilated_x[new_indices] = actions
                x_tensor = dilated_x
            else:
                new_x = torch.zeros_like(x_tensor)
                new_x[:n_actions] = actions
                x_tensor = new_x

        if self.has_labels:
            y_tensor = torch.tensor(self.y_data[idx], dtype=torch.float32)
            return x_tensor, y_tensor
        
        return x_tensor

In [33]:
def create_dataloaders(x_train, y_train, x_val, y_val, x_test, 
                       batch_size=BATCH_SIZE,
                       num_workers=2,
                       seed_worker=seed_worker,
                       data_generator=data_generator):
    
    train_dataset = UserBehaviorDataset(x_train, y_train, augment=False)
    val_dataset = UserBehaviorDataset(x_val, y_val, augment=False)
    test_dataset = UserBehaviorDataset(x_test, None, augment=False)

    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,           
        pin_memory=True,
        persistent_workers=False 
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,           
        pin_memory=True,
        persistent_workers=False 
    )
    
    test_loader = DataLoader(
        test_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader

In [34]:
import os
import sys
from typing import List
import torch
import torch.nn as nn
class AttentionPooling1D(nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.attn = nn.Linear(in_features, 1)

    def forward(self, x, mask):
        scores = self.attn(x) 
        
        if mask is not None:
            fill_value = torch.finfo(scores.dtype).min
            scores = scores.masked_fill(~mask.unsqueeze(-1), fill_value)
            
        attn_weights = F.softmax(scores, dim=1)
        return torch.sum(x * attn_weights, dim=1)

class ModernTCNBlock(nn.Module):
    def __init__(self, 
                 d_model: int, 
                 kernel_size: int, 
                 expansion_factor: int = 2, 
                 dropout: float = 0.2) -> None:
        """
        ModernTCN block với kiến trúc depthwise separable convolution
        
        Tham số:
            d_model: Số chiều của embedding đầu vào
            kernel_size: Kích thước kernel cho convolution
            expansion_factor: Hệ số mở rộng cho hidden layer trong block
            dropout: Tỷ lệ dropout cho các lớp fully connected
        """
        super(ModernTCNBlock, self).__init__()
        
        padding = kernel_size // 2
        self.dwconv = nn.Conv1d(in_channels=d_model, 
                                out_channels=d_model, 
                                kernel_size=kernel_size, 
                                padding=padding, 
                                groups=d_model)
        
        self.norm = nn.GroupNorm(num_groups=1, 
                                 num_channels=d_model)
        
        hidden_dim = int(d_model * expansion_factor)
        self.pwconv1 = nn.Conv1d(in_channels=d_model, 
                                 out_channels=hidden_dim, 
                                 kernel_size=1)
        self.act = nn.GELU()
        self.dropout1 = nn.Dropout(p=dropout)
        
        self.pwconv2 = nn.Conv1d(in_channels=hidden_dim, 
                                 out_channels=d_model, 
                                 kernel_size=1)
        self.dropout2 = nn.Dropout(p=dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        res = x
        x = self.dwconv(x)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.dropout1(x)
        x = self.pwconv2(x)
        x = self.dropout2(x)
        return x + res


In [35]:
class CascadeRegressionHead(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.head_start = nn.Sequential(
            nn.Linear(d_model, 64), 
            nn.GELU(), 
            nn.Linear(64, 3)
        )
        
        self.head_factory = nn.Sequential(
            nn.Linear(d_model, 128), 
            nn.LayerNorm(128), 
            nn.GELU(), 
            nn.Dropout(0.2), 
            nn.Linear(128, 64), 
            nn.GELU(), 
            nn.Linear(64, 2)
        )
        
        self.start_proj = nn.Sequential(
            nn.Linear(3, 16),
            nn.GELU(),
            nn.Linear(16, 32)
        )
        
        self.head_end = nn.Sequential(
            nn.Linear(d_model + 32, 64), 
            nn.GELU(), 
            nn.Linear(64, 3)
        )

    def forward(self, x):
        start_preds = self.head_start(x)  
        
        factory_preds = self.head_factory(x) 
        
        start_context = self.start_proj(start_preds)
        end_in = torch.cat([x, start_context], dim=-1)
        end_preds = self.head_end(end_in)    
        
        s_year = torch.sigmoid(start_preds[:, 0])
        e_year = torch.sigmoid(end_preds[:, 0])
        
        s_month = 1.0 + 11.0 * torch.sigmoid(start_preds[:, 1])
        e_month = 1.0 + 11.0 * torch.sigmoid(end_preds[:, 1])
        
        s_day = 1.0 + 30.0 * torch.sigmoid(start_preds[:, 2])
        e_day = 1.0 + 30.0 * torch.sigmoid(end_preds[:, 2])
        
        s_power = 99.0 * torch.sigmoid(factory_preds[:, 0])
        e_power = 99.0 * torch.sigmoid(factory_preds[:, 1])
        
        return [
            s_year, s_month, s_day, s_power,
            e_year, e_month, e_day, e_power
        ]

In [36]:
import torch
import torch.nn as nn
from typing import List


class TACNet(nn.Module):
    def __init__(self, 
                 vocab_size: int, 
                 seq_length: int = 37, 
                 embedding_dim: int = 256, 
                 kernel_sizes: List[int] = [3, 5, 7], 
                 expansion_factor: int = 2, 
                 dropout_rate: float = 0.3,
                 w2v_tensor: torch.Tensor = None) -> None: # Set default None cho w2v_tensor để linh hoạt
        
        super(TACNet, self).__init__()
        
        # 1. Embeddings
        self.embedding = nn.Embedding(num_embeddings=vocab_size, 
                                      embedding_dim=embedding_dim, 
                                      padding_idx=0)
        self.pos_embedding = nn.Embedding(num_embeddings=seq_length, 
                                          embedding_dim=embedding_dim)
        
        # 2. CNN Backbone (Modern TCN Blocks)
        blocks = []
        for ks in kernel_sizes:
            blocks.append(
                ModernTCNBlock(d_model=embedding_dim, 
                               kernel_size=ks, 
                               expansion_factor=expansion_factor, 
                               dropout=dropout_rate)
            )
        self.backbone = nn.Sequential(*blocks)
        
        # 3. Pooling & Dropout
        self.attn_pool = AttentionPooling1D(in_features=embedding_dim)
        self.dropout = nn.Dropout(p=dropout_rate)
        
        # --- THAY ĐỔI CHÍNH Ở ĐÂY ---
        # Sử dụng CascadeRegressionHead thay vì danh sách các Linear Classifier
        self.cascade_head = CascadeRegressionHead(d_model=embedding_dim)

        # Init weights
        self.apply(self._init_weights)
        if w2v_tensor is not None:
            self.embedding.weight.data.copy_(w2v_tensor) 

    def _init_weights(self, m: nn.Module) -> None:
        if isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv1d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.GroupNorm):
            nn.init.constant_(m.weight, 1.0)
            nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.1)
            if m.padding_idx is not None:
                nn.init.constant_(m.weight[m.padding_idx], 0)

    def forward(self, x: torch.Tensor) -> List[torch.Tensor]:
        batch_size, seq_len = x.size()
        
        # Tính mask
        mask = (x != 0) 
        
        # Tính Positional Embedding
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0).expand(batch_size, seq_len)
        pos_emb = self.pos_embedding(positions)
        pos_emb = pos_emb * mask.unsqueeze(-1).float() 
        
        # Input Embedding
        embedded = self.embedding(x) + pos_emb
        embedded = embedded * mask.unsqueeze(-1).float()
        
        # Đổi shape từ (B, L, C) sang (B, C, L) để phù hợp với Conv1d của TCN
        embedded = embedded.permute(0, 2, 1) 
        
        # Pass qua CNN Backbone
        features = self.backbone(embedded)    
        
        # Đổi shape ngược lại (B, C, L) về (B, L, C) để cho Attention Pooling
        features = features.permute(0, 2, 1) 
        
        # Pooling về vector (B, C)
        pooled_features = self.attn_pool(features, mask=mask)
        x_drop = self.dropout(pooled_features)
        
        # --- THAY ĐỔI CHÍNH Ở ĐÂY ---
        # Đưa context vector (B, C) qua CascadeRegressionHead
        # Trả về list 8 giá trị giống hệt AdvancedTAGNet
        return self.cascade_head(x_drop)

In [37]:
def post_process_predictions(preds_tensor):
    preds = preds_tensor.clone()
    preds = torch.round(preds)
    
    preds[:, [0, 3]] = (preds[:, [0, 3]] - 1) % 12 + 1
    preds[:, [0, 3]] = torch.clamp(preds[:, [0, 3]], 1, 12)

    days_in_month_t = torch.tensor(DAYS_IN_MONTH, device=preds.device, dtype=preds.dtype)
    
    for col_idx, month_idx in [(1, 0), (4, 3)]:
        m_indices = (preds[:, month_idx].long() - 1).clamp(0, 11)
        max_days = days_in_month_t[m_indices]
        
        preds[:, col_idx] = torch.clamp(preds[:, col_idx], min=1)
        preds[:, col_idx] = torch.min(preds[:, col_idx], max_days)

    preds[:, [2, 5]] = torch.clamp(preds[:, [2, 5]], 0, 99)
    
    return preds

In [38]:
def extract_hybrid_features(model, dataloader, device=DEVICE):
    model.eval()
    
    extracted_embeddings = []
    
    def hook_fn(module, input, output):
        extracted_embeddings.append(output.detach().cpu().numpy())
        
    handle = model.attn_pool.register_forward_hook(hook_fn)
    
    all_preds = []
    y_true_list = []
    
    print("Đang trích xuất đặc trưng...")
    with torch.no_grad():
        for batch in dataloader:
            if isinstance(batch, (list, tuple)) and len(batch) == 2:
                x, y = batch
                y_true_list.append(y.cpu().numpy())
            else:
                x = batch[0] if isinstance(batch, (list, tuple)) else batch
                
            x = x.to(device)
            
            outputs = model(x) 
            
            preds_batch = torch.stack(outputs, dim=1).cpu().numpy()
            all_preds.append(preds_batch)
            
    handle.remove()
    
    final_embeddings = np.concatenate(extracted_embeddings, axis=0)
    final_preds = np.concatenate(all_preds, axis=0)
    
    x_hybrid = np.concatenate([final_embeddings, final_preds], axis=1) 
    
    if len(y_true_list) > 0:
        y_true = np.concatenate(y_true_list, axis=0)
        return x_hybrid, y_true
    else:
        return x_hybrid

In [39]:
class SequenceFeatureExtractor:
    def __init__(self, top_k_vocab=50):
        self.vectorizer = CountVectorizer(max_features=top_k_vocab, token_pattern=r"(?u)\b\w+\b")
        self.is_fitted = False

    def extract(self, x_df, feature_cols=FEATURE_COLS):
        print(f"Đang trích xuất đặc cho {len(x_df)} mẫu...")
        seqs = x_df[feature_cols].values 
        
        seq_lengths = np.sum(seqs != 0, axis=1)
        consecutive_repeats = np.sum((seqs[:, 1:] == seqs[:, :-1]) & (seqs[:, 1:] != 0), axis=1)
        padding_ratio = (len(feature_cols) - seq_lengths) / len(feature_cols)
        
        num_uniques = np.zeros(len(seqs), dtype=np.float32)
        diversity = np.zeros(len(seqs), dtype=np.float32)
        first_codes = np.zeros(len(seqs), dtype=np.float32)
        last_codes = np.zeros(len(seqs), dtype=np.float32)
        
        most_frequent_codes = np.zeros(len(seqs), dtype=np.float32)
        obsession_ratios = np.zeros(len(seqs), dtype=np.float32)
        center_of_mass = np.zeros(len(seqs), dtype=np.float32)
        
        seq_strings = [] 
        for i, row in enumerate(seqs):
            non_zeros = row[row != 0]
            if len(non_zeros) > 0:
                vals, counts = np.unique(non_zeros, return_counts=True)
                num_uniques[i] = len(vals)
                diversity[i] = num_uniques[i] / len(non_zeros)
                first_codes[i] = non_zeros[0]
                last_codes[i] = non_zeros[-1] 
                
                best_idx = np.argmax(counts)
                most_frequent_codes[i] = vals[best_idx]
                obsession_ratios[i] = counts[best_idx] / len(non_zeros)
                
                center_of_mass[i] = np.mean(np.arange(len(non_zeros))) / len(non_zeros)
                
                seq_strings.append(" ".join(non_zeros.astype(str)))
            else:
                seq_strings.append("")

        if not self.is_fitted:
            bow_features = self.vectorizer.fit_transform(seq_strings).toarray()
            self.is_fitted = True
        else:
            bow_features = self.vectorizer.transform(seq_strings).toarray()
        
        manual_features = np.column_stack([
            seq_lengths,
            num_uniques,
            diversity,
            first_codes,
            last_codes,
            consecutive_repeats,
            padding_ratio,
            most_frequent_codes,
            obsession_ratios,
            center_of_mass,
            bow_features  
        ])
        
        return manual_features.astype(np.float32)

In [40]:
x_train = pd.read_csv(X_TRAIN)
y_train = pd.read_csv(Y_TRAIN)
x_val = pd.read_csv(X_VAL)
y_val = pd.read_csv(Y_VAL)
x_test = pd.read_csv(X_TEST)

x_train, y_train, x_val, y_val, x_test, VOCAB_SIZE = process_data(
    x_train, y_train, x_val, y_val, x_test
)

y_val_orig = (y_val[ATTRIBUTE_COLS].values)[:, [1, 2, 3, 5, 6, 7]] 

train_loader, val_loader, test_loader = create_dataloaders(
    x_train, y_train, x_val, y_val, x_test
)

Xử lý trùng lặp nội bộ...
 - Loại bỏ 99 hàng trùng lặp trong Train
 - Loại bỏ 110 hàng trùng lặp trong Val
Xử lý trùng lặp giữa Train và Val...
 - Loại bỏ 187 chuỗi Train bị trùng với Val
Xử lý lỗi ngày tháng...
 - Phát hiện 1010 hàng có lỗi ngày trong Train
 - Phát hiện 749 hàng có lỗi ngày trong Val
Xây dựng từ điển và map index...
 - Kích thước từ điển 934 (PAD=0, UNK=1)


In [41]:
extractor = SequenceFeatureExtractor(top_k_vocab=50)

x_train_manual = extractor.extract(x_train) 
x_val_manual = extractor.extract(x_val)
x_test_manual = extractor.extract(x_test)

Đang trích xuất đặc cho 54714 mẫu...
Đang trích xuất đặc cho 39890 mẫu...
Đang trích xuất đặc cho 108673 mẫu...


In [42]:
model = TACNet(vocab_size=VOCAB_SIZE, seq_length=SEQ_LENGTH).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

x_train_embed, y_train_embed = extract_hybrid_features(model, train_loader, device=DEVICE)
x_val_embed, y_val_embed = extract_hybrid_features(model, val_loader, device=DEVICE)
x_test_embed = extract_hybrid_features(model, test_loader, device=DEVICE)

RuntimeError: Error(s) in loading state_dict for TACNet:
	Missing key(s) in state_dict: "emb_norm.weight", "emb_norm.bias", "local_backbone.0.dwconv.weight", "local_backbone.0.dwconv.bias", "local_backbone.0.norm.weight", "local_backbone.0.norm.bias", "local_backbone.0.pwconv1.weight", "local_backbone.0.pwconv1.bias", "local_backbone.0.pwconv2.weight", "local_backbone.0.pwconv2.bias", "local_backbone.1.dwconv.weight", "local_backbone.1.dwconv.bias", "local_backbone.1.norm.weight", "local_backbone.1.norm.bias", "local_backbone.1.pwconv1.weight", "local_backbone.1.pwconv1.bias", "local_backbone.1.pwconv2.weight", "local_backbone.1.pwconv2.bias", "local_backbone.2.dwconv.weight", "local_backbone.2.dwconv.bias", "local_backbone.2.norm.weight", "local_backbone.2.norm.bias", "local_backbone.2.pwconv1.weight", "local_backbone.2.pwconv1.bias", "local_backbone.2.pwconv2.weight", "local_backbone.2.pwconv2.bias", "global_context.layers.0.self_attn.in_proj_weight", "global_context.layers.0.self_at

In [ ]:
x_train_final = np.concatenate([x_train_embed, x_train_manual], axis=1)
x_val_final = np.concatenate([x_val_embed, x_val_manual], axis=1)
x_test_final = np.concatenate([x_test_embed, x_test_manual], axis=1)

In [ ]:
ml_val_preds = np.zeros((len(x_val_final), 6), dtype=np.float32)
ml_test_preds = np.zeros((len(x_test_final), 6), dtype=np.float32)

TARGET_INDICES = [1, 2, 3, 5, 6, 7]
N_TRIALS = 10

def get_optuna_lgb_params(trial):
    return {
        'objective': 'regression',    
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 256),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'n_jobs': -1,                
        'seed': RANDOM_SEED,
        'verbose': -1,
        'feature_pre_filter': False
    }

for i, target_idx in enumerate(TARGET_INDICES):
    attr_name = ATTRIBUTE_LIST[i]
    
    y_train_target = y_train_embed[:, target_idx]
    y_val_target = y_val_embed[:, target_idx]
    
    train_data = lgb.Dataset(x_train_final, label=y_train_target)
    val_data = lgb.Dataset(x_val_final, label=y_val_target, reference=train_data)
    
    def objective_lgb(trial):
        params = get_optuna_lgb_params(trial)
        
        gbm_cv = lgb.train(
            params,
            train_data,
            num_boost_round=1000,
            valid_sets=[val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False)
            ]
        )
        return gbm_cv.best_score['valid_0']['rmse']

    study = optuna.create_study(
        direction="minimize", 
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED)
    )
    
    with tqdm(total=N_TRIALS, desc=f"Tuning Attr_{attr_name}") as pbar:
        def tqdm_callback(study, trial):
            pbar.update(1)
            pbar.set_postfix({"Best RMSE": f"{study.best_value:.4f}"})
            
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study.optimize(objective_lgb, n_trials=N_TRIALS, callbacks=[tqdm_callback])

    best_params = study.best_params
    best_params.update({
        'objective': 'regression', 
        'metric': 'rmse', 
        'boosting_type': 'gbdt', 
        'n_jobs': -1, 
        'seed': RANDOM_SEED, 
        'verbose': -1,
        'feature_pre_filter': False
    })
    
    print(f"Tham số tốt nhất cho Attr_{attr_name}: {study.best_params}")

    final_gbm = lgb.train(
        best_params,
        train_data,
        num_boost_round=2000, 
        valid_sets=[train_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=200)
        ]
    )
    
    print(f"Hoàn tất Attr_{attr_name} | Best Iteration: {final_gbm.best_iteration} | Final Val RMSE: {final_gbm.best_score['valid_1']['rmse']:.4f}")
    
    ml_val_preds[:, i] = final_gbm.predict(x_val_final, num_iteration=final_gbm.best_iteration)
    ml_test_preds[:, i] = final_gbm.predict(x_test_final, num_iteration=final_gbm.best_iteration)

In [ ]:
def evaluate_wmse(y_true, y_pred):
    M, W = np.array(M_LIST_METRIC), np.array(W_LIST_VALS)
    return np.mean(np.sum(W * np.power((y_true - y_pred) / M, 2), axis=1))
    
def wmape(y_t, y_p):
    sum_abs_error = np.sum(np.abs(y_t - y_p))
    sum_abs_true = np.sum(np.abs(y_t))
    return (sum_abs_error / (sum_abs_true + 1e-8)) * 100

print("\n" + "=" * 76)
print(f"{'THUỘC TÍNH':<12} | {'MAE':<10} | {'MSE':<12} | {'RMSE':<10} | {'WMAPE':<10}")
print("-" * 76)

ml_val_tensor = torch.tensor(ml_val_preds, device=DEVICE, dtype=torch.float32)
val_preds = post_process_predictions(ml_val_tensor).cpu().numpy()

for i, attr in enumerate(ATTRIBUTE_LIST):
    y_t = y_val_orig[:, i]
    y_p = val_preds[:, i]
    
    mae = np.mean(np.abs(y_t - y_p))
    mse = np.mean(np.square(y_t - y_p))
    rmse = np.sqrt(mse)
    wmape_val = wmape(y_t, y_p)
    
    print(f"Attr {attr:<7} | {mae:<10.4f} | {mse:<12.4f} | {rmse:<10.4f} | {wmape_val:<10.4f}")

print("=" * 76)

overall_mae = np.mean(np.abs(y_val_orig - val_preds))
overall_mse = np.mean(np.square(y_val_orig - val_preds))
overall_rmse = np.sqrt(overall_mse)
overall_wmape = wmape(y_val_orig, val_preds)
final_wmse = evaluate_wmse(y_val_orig, val_preds)

print(f"{'MAE':<15}: {overall_mae:.4f}")
print(f"{'MSE':<15}: {overall_mse:.4f}")
print(f"{'RMSE':<15}: {overall_rmse:.4f}")
print(f"{'WMAPE':<15}: {overall_wmape:.4f}")
print(f"{'WMSE':<15}: {final_wmse:.4f}")
print("=" * 76)

In [ ]:
ml_tensor = torch.tensor(ml_test_preds, device=DEVICE, dtype=torch.float32)
processed_ml_preds = post_process_predictions(ml_tensor)

test_preds = processed_ml_preds.cpu().numpy()
submission_df = pd.DataFrame({"id": x_test["id"]})

for idx, attr in enumerate(ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_preds[:, idx].astype(np.uint16)

submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)